# 03 — Celestial North from the background stars

The paper's 168° (celestial North, CCW from the image +x axis) was measured by
hand by overlaying a Stellarium chart on one frame (paper Fig. 5).  This
notebook re‑derives it from the data:

1. catalogue stars (Simbad, V < 8.5) within 1.8° of the Sun, cached in `data/reference_stars.csv`;
2. for several 1/3 s frames (the deepest exposures) of each polariser position: detect point
   sources away from the corona, match them to the catalogue at the nominal orientation,
   fit 2‑D Gaussians → centroids and PSF FWHM;
3. per frame, fit the North angle **α** and the plate scale to the measured positions
   (reference: the CHT Moon centre and the Skyfield topocentric Moon);
4. a translation‑free estimate of α from the *direction between star pairs*.

Outputs: `products/celestial_north_fit.csv`, `products/celestial_north_summary.csv`,
`figures/celestial_north_star_cutouts.png`, `figures/celestial_north_overlay.png`.
`config.CELESTIAL_NORTH_DEG` stays the constant used downstream; the summary
cell checks that the fit agrees with it.

Convention (array coordinates, x right, y down): North n = (cos α, −sin α), East
e = (−sin α, −cos α); a sky offset (sep, PA) maps to `sep · (sin PA · e + cos PA · n)` px.

In [ ]:
import sys, time
sys.path.insert(0, "..")          # config.py / utils.py live one level up
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits

import config, utils
%matplotlib inline

In [ ]:
import math
from scipy import ndimage

N_PER_POSITION = 2                    # 1/3 s frames analysed per polariser position (each ~40 s)
MATCH_RADIUS_PX = 40                  # catalogue -> detection association radius at the nominal orientation
CUT = 12                              # half-size of the fit cutout

eph = utils.Ephemeris()
cat = utils.reference_star_positions()
# zeta Psc A+B (23", unresolved in our ~10 px PSF): replace A and B by their flux-weighted blend
ab = cat[cat.name.isin(["zet Psc A", "zet Psc B"])]
if len(ab) == 2:
    w = 10 ** (-0.4 * ab.vmag.values); w /= w.sum()
    blend = dict(name="zet Psc AB", ra_deg=float((ab.ra_deg * w).sum()), dec_deg=float((ab.dec_deg * w).sum()),
                 vmag=float(-2.5 * np.log10((10 ** (-0.4 * ab.vmag)).sum())))
    cat = pd.concat([cat[~cat.name.isin(["zet Psc A", "zet Psc B"])], pd.DataFrame([blend])], ignore_index=True)
cat = cat.sort_values("vmag").reset_index(drop=True)
cat

In [ ]:
# Moon centres (source per config) and the frames to analyse
if config.MOON_CENTERS_SOURCE == "legacy":
    moon = pd.read_csv(config.LEGACY_MOON_CENTERS_CSV); moon["Filename"] = moon.Filename.str.split("/").str[-1]
else:
    moon = pd.read_csv(config.MOON_CENTERS_CSV)
moon = moon.drop_duplicates("Filename").set_index("Filename")
inv = pd.read_csv(config.PRODUCTS_DIR / "frame_inventory.csv", dtype={"position": str})
frames = (inv[(inv.inverse_exposure_time == 3) & inv.position.isin(config.POLARIZER_POSITIONS)]
          .sort_values("filename").groupby("position").head(N_PER_POSITION))
frames[["filename", "position", "date_obs_utc"]]

In [ ]:
def analyse_frame(f):
    """Detect + fit the catalogue stars in one frame."""
    L = utils.luminance(fits.getdata(config.CALIBRATED_LIGHTS_DIR / f).astype(np.float32))
    m = moon.loc[f]
    utc = utils.fits_timestamp_utc(utils.read_header(f))
    moon_r_as, _, _, _ = eph.sun_moon_geometry(utc)
    scale0 = moon_r_as / m.Moon_Radius                        # arcsec/px from the lunar radius
    off = eph.star_offsets(utc, cat, body="moon")
    det, hp = utils.detect_point_sources(L, center_xy=(m.Moon_XC, m.Moon_YC), nsigma=5, max_sources=40)
    rows, cuts = [], {}
    for (_, s), (_, o) in zip(cat.iterrows(), off.iterrows()):
        px, py = np.array([m.Moon_XC, m.Moon_YC]) + np.array(utils.sky_offset_to_pixels(o.sep_arcsec, o.pa_deg, 1 / scale0))
        inside = CUT < px < L.shape[1] - CUT and CUT < py < L.shape[0] - CUT
        d = det[np.hypot(det.x - px, det.y - py) < MATCH_RADIUS_PX] if inside else det.iloc[0:0]
        if d.empty:
            rows.append(dict(name=s["name"], vmag=s.vmag, pred_x=px, pred_y=py, detected=False, inside=inside)); continue
        x, y = int(d.iloc[0].x), int(d.iloc[0].y)
        cut = hp[y - CUT:y + CUT + 1, x - CUT:x + CUT + 1]
        try:
            fit = utils.fit_gaussian_2d(cut)
        except Exception:
            rows.append(dict(name=s["name"], vmag=s.vmag, pred_x=px, pred_y=py, detected=False, inside=inside)); continue
        rows.append(dict(name=s["name"], vmag=s.vmag, pred_x=px, pred_y=py, detected=True, inside=inside,
                         x=x - CUT + fit["x"], y=y - CUT + fit["y"], x_err=fit["x_err"], y_err=fit["y_err"],
                         fwhm_px=fit["fwhm"], snr=float(d.iloc[0].snr), sep_arcsec=o.sep_arcsec, pa_deg=o.pa_deg))
        cuts[s["name"]] = (cut, fit["model"])
    return pd.DataFrame(rows), cuts, m, utc, scale0

results, cutouts = {}, {}
for _, r in frames.iterrows():
    t0 = time.time()
    tab, cuts, m, utc, scale0 = analyse_frame(r.filename)
    results[r.filename] = dict(table=tab, moon=m, utc=utc, scale0=scale0, position=r.position)
    cutouts[r.filename] = cuts
    print(f"{r.filename}  pos{r.position}  {utc[11:19]}  scale0={scale0:.4f}\"/px  detected: {list(tab[tab.detected].name)}  ({time.time()-t0:.0f} s)")

In [ ]:
# Per-frame orientation fits
rows = []
for f, R in results.items():
    ok = R["table"][R["table"].detected].copy()
    if len(ok) < 2:
        print(f"{f}: fewer than 2 stars — skipped"); continue
    m = R["moon"]
    # (a) alpha and plate scale with the Moon centre fixed
    sol = utils.fit_image_rotation_to_stars(ok[["x", "y"]].values, ok, center_xy=(m.Moon_XC, m.Moon_YC), scale0=R["scale0"])
    # (b) translation-free alpha from the direction between the two brightest stars
    a, b = ok.iloc[0], ok.iloc[1]
    meas_ang = math.degrees(math.atan2(b.y - a.y, b.x - a.x))
    pa_, pb_ = [np.array(utils.sky_offset_to_pixels(s.sep_arcsec, s.pa_deg, 1 / R["scale0"])) for s in (a, b)]
    pred_ang = math.degrees(math.atan2(pb_[1] - pa_[1], pb_[0] - pa_[0]))
    alpha_pair = config.CELESTIAL_NORTH_DEG - (meas_ang - pred_ang)     # rotating the image by +d rotates alpha by -d (y down)
    scale_pair = np.hypot(*(pb_ - pa_)) * R["scale0"] / np.hypot(b.x - a.x, b.y - a.y)
    # (c) mean star-based offset of the Moon centre at the nominal orientation
    dx = (ok.x - ok.pred_x).mean(); dy = (ok.y - ok.pred_y).mean()
    rows.append(dict(filename=f, position=R["position"], utc=R["utc"], n_stars=len(ok), stars=" | ".join(ok.name),
                     alpha_fit=sol["alpha_deg"], alpha_fit_err=sol["alpha_err"], scale_fit=sol["scale"], scale_fit_err=sol["scale_err"],
                     hand=sol["hand"], fit_rms_px=sol["rms_px"], alpha_pair=alpha_pair, scale_pair=scale_pair,
                     scale_from_moon_radius=R["scale0"], moon_offset_dx=dx, moon_offset_dy=dy,
                     fwhm_px=ok.fwhm_px.mean(), min_snr=ok.snr.min()))
fit = pd.DataFrame(rows)
fit.to_csv(config.CELESTIAL_NORTH_FIT_CSV, index=False)
fit.round(3)

In [ ]:
summary = dict(
    alpha_fit_mean=fit.alpha_fit.mean(), alpha_fit_std=fit.alpha_fit.std(ddof=1),
    alpha_pair_mean=fit.alpha_pair.mean(), alpha_pair_std=fit.alpha_pair.std(ddof=1),
    scale_fit_mean=fit.scale_fit.mean(), scale_pair_mean=fit.scale_pair.mean(), scale_moon_mean=fit.scale_from_moon_radius.mean(),
    moon_offset_dx=fit.moon_offset_dx.mean(), moon_offset_dy=fit.moon_offset_dy.mean(),
    moon_offset_dx_std=fit.moon_offset_dx.std(ddof=1), moon_offset_dy_std=fit.moon_offset_dy.std(ddof=1),
    fit_rms_px=fit.fit_rms_px.mean(),
    fwhm_px=fit.fwhm_px.mean(), hand=int(fit.hand.mode()[0]))
print(f"North angle from fits (Moon fixed):      {summary['alpha_fit_mean']:.2f} ± {summary['alpha_fit_std']:.2f} deg")
print(f"North angle from star-pair direction:    {summary['alpha_pair_mean']:.2f} ± {summary['alpha_pair_std']:.2f} deg   (translation-free)")
print(f"config.CELESTIAL_NORTH_DEG (Stellarium): {config.CELESTIAL_NORTH_DEG:.2f} deg")
print(f"plate scale: fit {summary['scale_fit_mean']:.4f}, star pair {summary['scale_pair_mean']:.4f}, lunar radius {summary['scale_moon_mean']:.4f} arcsec/px  (paper: {config.PLATE_SCALE_ARCSEC_PER_PIX})")
print(f"handedness that fits the sky: {summary['hand']:+d}  (+1 = the convention in config.py)")
print(f"rms residual of the stars after fitting (alpha, scale) with the Moon fixed: {summary['fit_rms_px']:.1f} px  "
      f"-> the CHT Moon centre agrees with the star field at that level")
print(f"(at the per-frame lunar-radius scale the stars appear offset by ({summary['moon_offset_dx']:+.1f}, {summary['moon_offset_dy']:+.1f}) px: "
      f"that is the ~0.5 % CHT-radius bias, {100*(summary['scale_moon_mean']/summary['scale_pair_mean']-1):+.2f} %, not a centre error; see docs/DISCREPANCIES.md)")
print(f"mean PSF FWHM: {summary['fwhm_px']:.1f} px = {summary['fwhm_px']*summary['scale_moon_mean']:.1f} arcsec")
pd.DataFrame([summary]).to_csv(config.PRODUCTS_DIR / "celestial_north_summary.csv", index=False)
assert summary["hand"] == 1, "the sky handedness does not match config.north_east_vectors"
assert abs(summary["alpha_pair_mean"] - config.CELESTIAL_NORTH_DEG) < 1.5, "North angle disagrees with the paper value by > 1.5 deg"

In [ ]:
# Cutouts: data | model | residual for the stars of the first frame
f0 = next(iter(cutouts)); cuts = cutouts[f0]
fig, axes = plt.subplots(len(cuts), 3, figsize=(7, 2.3 * len(cuts)), squeeze=False)
for row, (name, (cut, model)) in zip(axes, cuts.items()):
    vmax = np.nanmax(cut)
    for ax, img, ttl in zip(row, (cut, model, cut - model), ("data", "Gaussian fit", "residual")):
        ax.imshow(img, cmap="magma", vmin=-0.2 * vmax, vmax=vmax); ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"{name}: {ttl}" if ttl == "data" else ttl, fontsize=9)
fig.suptitle(f"{f0}", fontsize=10); plt.tight_layout()
fig.savefig(config.FIGURES_DIR / "celestial_north_star_cutouts.png", dpi=150, bbox_inches="tight")

In [ ]:
# Overlay: the frame with predicted (circles) and measured (crosses) stars, N/E arrows at the Moon, solar North
R = results[f0]; m = R["moon"]; tab = R["table"]
img = fits.getdata(config.CALIBRATED_LIGHTS_DIR / f0, memmap=True)
L8 = utils.luminance(np.asarray(img[:, ::4, ::4], dtype=np.float32))
fig, ax = plt.subplots(figsize=(13, 8.7))
ax.imshow(utils.asinh_stretch(L8, 5, 99.9), cmap="gray", extent=(0, config.IMAGE_SHAPE[1], config.IMAGE_SHAPE[0], 0))
for _, s in tab[tab.inside].iterrows():
    ax.add_patch(plt.Circle((s.pred_x, s.pred_y), 45, ec="cyan", fc="none", lw=1.2))
    ax.annotate(f"{s['name']} (V={s.vmag:.1f})", (s.pred_x, s.pred_y), xytext=(55, -10), textcoords="offset points", color="cyan", fontsize=9)
    if s.detected:
        ax.plot(s.x, s.y, "+", color="yellow", ms=14, mew=1.5)
n, e = utils.north_east_vectors(config.CELESTIAL_NORTH_DEG)
sn, _ = utils.north_east_vectors(config.SOLAR_NORTH_ANGLE_DEG)
for vec, lab, col in ((n, "N (celestial)", "lime"), (e, "E", "lime"), (sn, "N (solar)", "orange")):
    ax.annotate("", xy=(m.Moon_XC + 900 * vec[0], m.Moon_YC + 900 * vec[1]), xytext=(m.Moon_XC, m.Moon_YC),
                arrowprops=dict(arrowstyle="->", color=col, lw=2))
    ax.text(m.Moon_XC + 1000 * vec[0], m.Moon_YC + 1000 * vec[1], lab, color=col, fontsize=11, fontweight="bold", ha="center")
ax.add_patch(plt.Circle((m.Moon_XC, m.Moon_YC), m.Moon_Radius, ec="deepskyblue", fc="none", lw=0.8, ls="--"))
ax.set_title(f"{f0} — catalogue predictions at α = {config.CELESTIAL_NORTH_DEG}° (cyan) vs measured centroids (yellow)", fontsize=11)
ax.set_xlabel("x [px]"); ax.set_ylabel("y [px]")
fig.savefig(config.FIGURES_DIR / "celestial_north_overlay.png", dpi=150, bbox_inches="tight")